# Movie Review Sentiment Analysis (NLP)
### Predicting positive vs negative sentiment from 50,000 IMDB reviews

## Introduction
Can a model read a movie review and tell whether it's positive or negative?
This project builds an NLP sentiment classifier, covering the full text
pipeline: cleaning raw text, converting it to numbers with TF-IDF, and
classifying it - then examining which words drive the model's decisions.

## Key Results
- 89.5% accuracy, 0.962 ROC-AUC on 10,000 held-out reviews
- Balanced performance across positive and negative classes

## Key Findings
- The strongest positive signals: "great", "excellent", "perfect", "amazing"
- The strongest negative signals: "worst", "awful", "bad", "boring"
- Negative words carried higher weights than positive ones, suggesting
  unhappy reviewers use more emphatic, decisive language
- Two-word phrases ("the best", "the worst") were captured thanks to
  bigram features, preserving context single words would miss

## NLP Techniques Used
- Text cleaning with regular expressions (HTML removal, lowercasing)
- TF-IDF vectorization with unigrams and bigrams
- Logistic Regression for high-dimensional text classification
- Model interpretation via coefficient analysis

## Tools Used
- Python, pandas, NumPy, scikit-learn, matplotlib
- Dataset: IMDB 50K Movie Reviews (Kaggle)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('IMDB Dataset.csv')

print(df.shape)
print(df.head())
print(df['sentiment'].value_counts())

(50000, 2)
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [2]:
# Look at one full review
print("--- A POSITIVE review ---")
print(df[df['sentiment'] == 'positive']['review'].iloc[0][:500])
# [:500] — first 500 characters so it doesn't flood the screen

print("\n--- A NEGATIVE review ---"),
print(df[df['sentiment'] == 'negative']['review'].iloc[0][:500])

--- A POSITIVE review ---
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ

--- A NEGATIVE review ---
Basically there's a family where a little boy (Jake) thinks there's a zombie in his closet & his parents are fighting all the time.<br /><br />This movie is slower than a soap opera... and suddenly, Jake decides to become Rambo and kill the zombie.<br /><br />OK, first of all when you're going to make a film you must Decide if its a thriller or a drama! As a drama the movie is watchable. Parents are divorcing & arguing like in real life. And 

In [3]:
import re
# re — Python's built-in regular expressions library
# used for find-and-replace patterns in text (like removing HTML tags)

def clean_text(text):
    """Clean a single review for analysis."""
    text = text.lower()
    # lowercase everything so 'Great' and 'great' are treated the same
    text = re.sub(r'<.*?>', ' ', text)
     # re.sub(pattern, replacement, text) — find and replace
    # '<.*?>' — matches any HTML tag like <br /> and replaces with a space
    text = re.sub(r'[^a-z\s]', ' ', text)
    # removes anything that isn't a letter or space (punctuation, numbers)
    text = re.sub(r'\s+', ' ', text).strip()
    # collapses multiple spaces into one, trims the ends
    return text

# apply cleaning to every review
df['clean_review'] = df['review'].apply(clean_text)
# .apply() — runs the function on every row of the column

# Compare before and after
print("BEFORE:")
print(df['review'].iloc[0][:200])
print("\nAFTER:")
print(df['clean_review'].iloc[0][:200])

BEFORE:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo

AFTER:
one of the other reviewers has mentioned that after watching just oz episode you ll be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its br


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# Convert sentiment labels to numbers
df['label'] = (df['sentiment'] == 'positive').astype(int)
# positive becomes 1, negative becomes 0

# Split into train and test FIRST (before vectorizing)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['clean_review'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label'])

# we split the raw text first, then vectorize
# this prevents the test set from influencing the vocabulary (data leakage)

# TF-IDF Vectorization — turns text into numbers
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
# max_features=5000 — keep only the 5000 most useful words/phrases
# ngram_range=(1,2) — consider single words AND two-word phrases
#   so "not good" is captured as a phrase, not just "not" and "good" separately

X_train = vectorizer.fit_transform(X_train_text)
# fit_transform on training data — learns the vocabulary AND converts to numbers
X_test = vectorizer.transform(X_test_text)
# transform on test data — uses the SAME vocabulary, doesn't re-learn

print(f"Training shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"\nVocabulary size: {len(vectorizer.vocabulary_)}")

Training shape: (40000, 5000)
Test shape: (10000, 5000)

Vocabulary size: 5000


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

#logistic regression works extremely well for text classification
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

print(f"Accuracy: {acc:.1%}")
print(f"ROC-AUC:  {auc:.3f}")
print("\n--- Classification Report ---")
print(classification_report(y_test, preds, target_names=['Negative', 'Positive']))

Accuracy: 89.5%
ROC-AUC:  0.962

--- Classification Report ---
              precision    recall  f1-score   support

    Negative       0.90      0.89      0.89      5000
    Positive       0.89      0.90      0.90      5000

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



In [7]:
import numpy as np

# Get the vocabulary and the model's learned weight for each word
feature_names = np.array(vectorizer.get_feature_names_out())
# get_feature_names_out() — the actual words/phrases behind each of the 5000 columns
coefficients = model.coef_[0]
# the model's learned weight for each word
# positive weight = pushes toward positive sentiment, negative = toward negative

# Most POSITIVE words
top_positive = np.argsort(coefficients)[-15:]
# argsort — returns indices that would sort the array
# [-15:] takes the 15 highest (most positive) weights

# Most NEGATIVE words
top_negative = np.argsort(coefficients)[:15]
# [:15] takes the 15 lowest (most negative) weights

print("--- Words that most signal POSITIVE sentiment ---")
for i in reversed(top_positive):
    print(f"  {feature_names[i]:20s} {coefficients[i]:.2f}")

print("\n--- Words that most signal NEGATIVE sentiment ---")
for i in top_negative:
    print(f"  {feature_names[i]:20s} {coefficients[i]:.2f}")

--- Words that most signal POSITIVE sentiment ---
  great                7.38
  excellent            6.40
  perfect              5.27
  amazing              5.07
  wonderful            4.89
  hilarious            4.33
  brilliant            4.33
  fun                  4.06
  enjoyable            4.00
  loved                3.95
  the best             3.92
  today                3.89
  superb               3.89
  enjoyed              3.64
  definitely           3.61

--- Words that most signal NEGATIVE sentiment ---
  worst                -8.31
  awful                -8.11
  bad                  -7.95
  boring               -6.82
  waste                -6.39
  poor                 -6.20
  the worst            -6.18
  terrible             -6.01
  dull                 -5.33
  poorly               -5.18
  horrible             -5.01
  worse                -4.88
  nothing              -4.84
  stupid               -4.61
  disappointing        -4.57
